# 🗄️ 01_carga_sqlserver — Carga de `ecommerce_pedidos` no SQL Server
**Squad 2 · Dupla 3** · Passo 4 do plano

Grava em `squad2.ecommerce_pedidos` um **espelho fiel** de todos os micro-lotes do `raw/real-time-data`: sem limpeza (papel da Silver),
apenas com duas colunas de rastreabilidade (`arquivo_origem` e `dt_carga`). Substitui a versão anterior da tabela, que misturava o histórico batch.

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração e contrato publicado pela EDA |
| 2 | Leitura de todos os micro-lotes |
| 3 | Preparação e impressão digital da origem |
| 4 | Gravação no SQL Server (`overwrite`) |
| 5 | Conferência lendo de volta do banco |
| 6 | Registro da carga em `metadata/` |

Cada etapa **faz** o trabalho e **valida** em seguida: ✅ ok · ⚠️ achado (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` em modo rápido e o contrato publicado pela EDA (`eda_raw.json`), para validar o schema com os mesmos tipos.
**Valida:** se as funções necessárias existem; sem contrato, segue validando só as colunas obrigatórias.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from pyspark.sql import functions as F

TABELA_SQL = nome_tabela_sql(TABELA)                    # squad2.ecommerce_pedidos
COLUNAS_RASTREIO = ["arquivo_origem", "dt_carga"]       # únicas colunas acrescentadas ao dado do raw
contrato_eda = ler_metadata("eda_raw")                  # schema e veredito publicados pela EDA (Etapa 9.2)
SCHEMA_ESPERADO = (contrato_eda or {}).get("schema", {})

# ------------------------------ validação ------------------------------
checar(all(n in globals() for n in ["ler_micro_lotes", "escrever_sql", "ler_sql_query"]), "Funções do 00_setup_config disponíveis",
       "Rode o %run ./00_setup_config atualizado (com ler_micro_lotes)")
checar(bool(SCHEMA_ESPERADO), f"Contrato da EDA carregado (veredito: {(contrato_eda or {}).get('veredito')})",
       "Contrato da EDA não encontrado: validando só as colunas obrigatórias", parar=False)
print(f"Destino: {TABELA_SQL}")

✅ Funções do 00_setup_config disponíveis
✅ Contrato da EDA carregado (veredito: APROVADA COM CONDIÇÕES)
Destino: squad2.ecommerce_pedidos


## Etapa 2 — Leitura dos micro-lotes
**Faz:** lista e lê **todos** os micro-lotes de pedidos do `raw`, numa única conversão para Spark.
**Valida:** todos os arquivos lidos, colunas obrigatórias (regra 1) e tipos conforme o contrato da EDA.

In [0]:
arquivos = listar_arquivos_origem()
df_raw = ler_micro_lotes(arquivos)

# ------------------------------ validação ------------------------------
checar(len(arquivos) > 0, f"{len(arquivos)} micro-lotes encontrados", "Nenhum micro-lote de pedidos no raw")
checar(df_raw.select("arquivo_origem").distinct().count() == len(arquivos), "Todos os micro-lotes foram lidos", "Algum micro-lote não foi lido")
tipos = dict(df_raw.dtypes)
faltam = [c for c in COLUNAS_OBRIGATORIAS if c not in tipos]
checar(not faltam, "Regra 1: colunas obrigatórias presentes", f"Regra 1: faltam {faltam}")
divergentes = {c: (t, tipos.get(c, "ausente")) for c, t in SCHEMA_ESPERADO.items() if tipos.get(c) != t}
checar(not divergentes, "Tipos conforme o contrato da EDA", f"Tipos divergentes (esperado, encontrado): {divergentes}", parar=False)

✅ 196 micro-lotes encontrados
✅ Todos os micro-lotes foram lidos
✅ Regra 1: colunas obrigatórias presentes
✅ Tipos conforme o contrato da EDA


## Etapa 3 — Preparação e impressão digital
**Faz:** acrescenta `dt_carga` (o `arquivo_origem` já vem da leitura), ordena as colunas e calcula a **impressão digital** da origem: linhas, pedidos distintos, arquivos, soma de `valor_total` e período. Se qualquer linha se perder ou um valor mudar no caminho até o banco, a impressão digital muda.
**Valida:** nenhuma linha perdida na preparação e unicidade de `id_pedido`.

In [0]:
colunas_dado = list(SCHEMA_ESPERADO) or [c for c in df_raw.columns if c not in COLUNAS_RASTREIO]
df_carga = df_raw.withColumn("dt_carga", F.current_timestamp()).select(*colunas_dado, *COLUNAS_RASTREIO)


def impressao_digital(df):
    """Resumo que identifica o conteúdo da tabela: tem que ser igual antes e depois da gravação."""
    return df.agg(F.count("*").alias("linhas"), F.countDistinct("id_pedido").alias("pedidos"),
                  F.countDistinct("arquivo_origem").alias("arquivos"), F.round(F.sum("valor_total"), 2).alias("soma_valor_total"),
                  F.date_format(F.min("dt_pedido"), "yyyy-MM-dd HH:mm:ss").alias("inicio"),
                  F.date_format(F.max("dt_pedido"), "yyyy-MM-dd HH:mm:ss").alias("fim")).first().asDict()


origem = impressao_digital(df_carga)

# ------------------------------ validação ------------------------------
checar(origem["arquivos"] == len(arquivos), f"{origem['linhas']:,} linhas de {origem['arquivos']} micro-lotes prontas para a carga",
       "Micro-lotes perdidos na preparação")
checar(origem["linhas"] == origem["pedidos"], "id_pedido único (o espelho pode ser carregado sem duplicidade)",
       f"{origem['linhas'] - origem['pedidos']} id_pedido repetidos: carregados como vieram (a planilha não pede deduplicação por pedido; a Silver monitora)", parar=False)

# ------------------------------ resultado ------------------------------
display(pd.DataFrame([origem]))

✅ 1,240 linhas de 196 micro-lotes prontas para a carga
✅ id_pedido único (o espelho pode ser carregado sem duplicidade)


linhas,pedidos,arquivos,soma_valor_total,inicio,fim
1240,1240,196,330209.5,2026-09-21 11:21:48,2026-10-07 10:55:55


## Etapa 4 — Gravação no SQL Server
**Faz:** grava em `squad2.ecommerce_pedidos` com `overwrite` (a tabela é recriada com o conteúdo atual do raw), pelo conector `sqlserver` do `00_setup_config`.
**Valida:** a conferência completa é a Etapa 5; aqui, só a conclusão da gravação.

In [0]:
escrever_sql(df_carga, TABELA, modo="overwrite")       # imprime ✅ ao concluir; qualquer falha interrompe a célula

✅ SQL Server: squad2.ecommerce_pedidos gravada (overwrite)


## Etapa 5 — Conferência no banco
**Faz:** lê a tabela **de volta do SQL Server** e compara a impressão digital com a da origem; confere também as colunas gravadas e mostra uma amostra.
**Valida:** linhas, pedidos, arquivos, soma e período iguais (a soma aceita diferença de arredondamento de até R$ 0,01).

In [0]:
banco = ler_sql_query(f"""
    SELECT COUNT(*) AS linhas, COUNT(DISTINCT id_pedido) AS pedidos, COUNT(DISTINCT arquivo_origem) AS arquivos,
           ROUND(SUM(valor_total), 2) AS soma_valor_total,
           FORMAT(MIN(dt_pedido), 'yyyy-MM-dd HH:mm:ss') AS inicio, FORMAT(MAX(dt_pedido), 'yyyy-MM-dd HH:mm:ss') AS fim
    FROM {TABELA_SQL}""").first().asDict()
colunas_banco = ler_sql_query(f"SELECT TOP 0 * FROM {TABELA_SQL}").columns

conferencia = pd.DataFrame([(k, str(origem[k]), str(banco[k])) for k in origem], columns=["medida", "origem", "sql_server"])
conferencia["confere"] = [abs(float(origem[k]) - float(banco[k])) <= 0.01 if k == "soma_valor_total" else origem[k] == banco[k] for k in origem]

# ------------------------------ validação ------------------------------
for _, linha in conferencia.iterrows():
    checar(linha["confere"], f"{linha['medida']}: {linha['sql_server']} (igual à origem)",
           f"{linha['medida']}: origem {linha['origem']} × banco {linha['sql_server']}")
checar(set(colunas_banco) == set(df_carga.columns), f"{len(colunas_banco)} colunas gravadas, iguais às enviadas",
       f"Colunas diferentes no banco: {set(colunas_banco) ^ set(df_carga.columns)}")

# ------------------------------ resultado ------------------------------
display(conferencia)
display(ler_sql_query(f"SELECT TOP 5 * FROM {TABELA_SQL} ORDER BY dt_pedido DESC"))

✅ linhas: 1240 (igual à origem)
✅ pedidos: 1240 (igual à origem)
✅ arquivos: 196 (igual à origem)
✅ soma_valor_total: 330209.5 (igual à origem)
✅ inicio: 2026-09-21 11:21:48 (igual à origem)
✅ fim: 2026-10-07 10:55:55 (igual à origem)
✅ 12 colunas gravadas, iguais às enviadas


medida,origem,sql_server,confere
linhas,1240,1240,true
pedidos,1240,1240,true
arquivos,196,196,true
soma_valor_total,330209.5,330209.5,true
inicio,2026-09-21 11:21:48,2026-09-21 11:21:48,true
fim,2026-10-07 10:55:55,2026-10-07 10:55:55,true


id_pedido,id_cliente,id_endereco_entrega,dt_pedido,status_pedido,valor_total,valor_frete,metodo_pagamento,dt_previsao_entrega,dt_ultima_atualizacao_status,arquivo_origem,dt_carga
1791381360,1791381360,1791381360,2026-10-07T10:55:55.053Z,Pagamento Aprovado,221.38,14.28,cartão crédito,2026-10-10T10:55:55.053Z,2026-10-07T10:55:55.053Z,real-time-data/2026/10/07/105607/ecommerce_pedidos.parquet,2026-10-08T12:19:17.090Z
1791381362,8222,698,2026-10-07T10:55:48.053Z,Processando,180.84,20.1,pix,2026-10-11T10:55:48.053Z,2026-10-07T10:55:48.053Z,real-time-data/2026/10/07/105607/ecommerce_pedidos.parquet,2026-10-08T12:19:17.090Z
1791381363,1791381360,1791381360,2026-10-07T10:55:31.053Z,Processando,218.77,20.6,cartão crédito,2026-10-15T10:55:31.053Z,2026-10-07T10:55:31.053Z,real-time-data/2026/10/07/105607/ecommerce_pedidos.parquet,2026-10-08T12:19:17.090Z
1791381364,9229,2313,2026-10-07T10:55:22.057Z,Processando,81.15,16.64,pix,2026-10-12T10:55:22.057Z,2026-10-07T10:55:22.057Z,real-time-data/2026/10/07/105607/ecommerce_pedidos.parquet,2026-10-08T12:19:17.090Z
1791381361,7844,473,2026-10-07T10:55:19.053Z,Pagamento Aprovado,127.18,15.42,cartão débito,2026-10-12T10:55:19.053Z,2026-10-07T10:55:19.053Z,real-time-data/2026/10/07/105607/ecommerce_pedidos.parquet,2026-10-08T12:19:17.090Z


## Etapa 6 — Registro da carga
**Faz:** grava em `metadata/ecommerce_pedidos/carga_sqlserver.json` quando a carga rodou e a impressão digital conferida (evidência do Passo 4) e libera a memória.
**Valida:** releitura do registro.

In [0]:
registro = {"executado_em_utc": datetime.now(timezone.utc).isoformat(), "tabela": TABELA_SQL, "modo": "overwrite",
            "impressao_digital": origem, "conferencia_ok": bool(conferencia["confere"].all())}
escrever_metadata("carga_sqlserver", registro)

# ------------------------------ validação ------------------------------
checar((ler_metadata("carga_sqlserver") or {}).get("executado_em_utc") == registro["executado_em_utc"],
       f"Carga registrada em {CONTAINER_SQUAD}/{caminho_metadata('carga_sqlserver')}", "Registro da carga não foi relido igual")

for _v in ["df_raw", "df_carga"]:
    globals().pop(_v, None)
print(f"🗄️ {TABELA_SQL} carregada: {origem['linhas']:,} pedidos de {origem['arquivos']} micro-lotes ({origem['inicio']} a {origem['fim']})")

✅ Carga registrada em squad2/metadata/ecommerce_pedidos/carga_sqlserver.json
🗄️ squad2.ecommerce_pedidos carregada: 1,240 pedidos de 196 micro-lotes (2026-09-21 11:21:48 a 2026-10-07 10:55:55)
